# Notebook for topic modeling 

# 0. Imports

In [1]:
## load packages
import pandas as pd
import re
import numpy as np

## nltk imports
#!pip install nltk # can install on terminal or by uncommenting this line
# import nltk; nltk.download('punkt'); nltk.download('stopwords')
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import nltk

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
#!pip install gensim # can install by uncommenting this line
from gensim import corpora
import gensim

## visualizing LDA--likely need to install
#!pip install pyLDAvis # can install by uncommenting this line
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

pyLDAvis.enable_notebook()

## print mult things
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

## random
import random
import string

punctlist = [char for char in string.punctuation]  # list of english punctuation marks

# 0. Load data

In [2]:
ab = pd.read_csv("../public_data/airbnb_text.zip")
ab.head()

,id,name,name_upper,neighbourhood_group,price
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80


# 1. Preprocess documents

In this case, each name/name_upper, or listing title, we're treating as a document

## 1.1 Load stopwords list and augment with our own custom ones

In [3]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /Users/rk/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
list_stopwords = stopwords.words("english")

custom_words_toadd = [
    "apartment",
    "new york",
    "nyc",
    "bronx",
    "brooklyn",
    "manhattan",
    "queens",
    "staten island",
]

list_stopwords_new = list_stopwords + custom_words_toadd

## 1.2 Remove stopwords from lowercase version of corpus


In [5]:
## convert to lowercase and a list
corpus_lower = ab.name.str.lower().to_list()
corpus_lower[0:5]

## use wordpunct tokenize and filter out with one
example_listing = corpus_lower[3]
nostop_listing = [
    word
    for word in wordpunct_tokenize(example_listing)
    if word not in list_stopwords_new
]
nostop_listing

['clean & quiet apt home by the park',
 'skylit midtown castle',
 'the village of harlem....new york !',
 'cozy entire floor of brownstone',
 'entire apt: spacious studio/loft by central park']

['cozy', 'entire', 'floor', 'brownstone']

## 1.3 stem and remove non-alpha

Other contexts we may want to leave digits in

In [6]:
## initialize stemmer
porter = PorterStemmer()

## apply to one by iterating
## over the tokens in the list
example_listing_preprocess = [
    porter.stem(token) for token in nostop_listing if token.isalpha() and len(token) > 2
]

example_listing_preprocess

['cozi', 'entir', 'floor', 'brownston']

In [7]:
example_listing
example_listing_preprocess

'cozy entire floor of brownstone'

['cozi', 'entir', 'floor', 'brownston']

## 1.4 Activity 1

The above example performed preprocessing on a single Airbnb listing. We want to generalize this preprocessing across all listings.

- Embed step two (remove stopwords) and step three (stem) into one or two functions that take in a raw string (eg the raw text of an Airbnb review) and return a preprocessed string 
- Apply the function iteratively to preprocess all the texts in `corpus_lower`. Output could either be a list where each list element is a string of a list (e.g., `cozy brownstone apt`), or a list of lists where each element is a tokenized string (e.g., `['cozy', 'brownstone', 'apt'])`

Output is flexible: it could be a list of lists containing tokenized/stemmed text or a list of strings.

In [8]:
stopword_tokens = {
    token
    for stopword in list_stopwords_new
    for token in wordpunct_tokenize(stopword.lower())
    if token.isalpha()
}


def preprocess_text(raw_text):
    """Return one text with stopwords removed and remaining words stemmed."""
    if not isinstance(raw_text, str):
        return ""

    tokens = wordpunct_tokenize(raw_text.lower())
    processed_tokens = [
        porter.stem(token)
        for token in tokens
        if token.isalpha() and len(token) > 2 and token not in stopword_tokens
    ]
    return " ".join(processed_tokens)


def preprocess_corpus(texts):
    """Apply preprocess_text to every document in an iterable."""
    return [preprocess_text(text) for text in texts]


corpus_preprocessed = preprocess_corpus(corpus_lower)
corpus_preprocessed[:5]

['clean quiet apt home park',
 'skylit midtown castl',
 'villag harlem',
 'cozi entir floor brownston',
 'entir apt spaciou studio loft central park']

# 2. Create a document-term matrix and do some basic diagnostics (more manual approach)

Here we'll create a DTM first using the raw documents; in the activity, you'll create one using the preprocessed docs
that you created in activity 1

## 2.1 Define the dtm function and select data to transform into a document-term matrix

In [9]:
## function provided
def create_dtm(list_of_strings, metadata):
    """
    Function to create dense document-term matrix (DTM) from a list of strings and provided metadata.
    A sparse DTM is a list of term_index/doc_index tuples: if a given term occurs in a given doc at least once,
        then this count is listed as a tuple; if not, that term/doc pair is omitted.
    In a dense DTM, each row is one text (e.g., an Airbnb listing), each column is a term, and
        each cell indicates the frequency of that word in that text.

    Parameters:
        list_of_strings (Series): each row contains a preprocessed string (need not be tokenized)
        metadata (DataFrame): contains document-level covariates

    Returns:
        Dense DTM with metadata on left and then one column per word in lexicon
    """

    # initialize a sklearn tokenizer; this helps us tokenize the preprocessed string input
    vectorizer = CountVectorizer(lowercase=True)
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    print(
        "Sparse matrix form:\n", dtm_sparse[:3]
    )  # take a look at sparse representation
    print()

    # switch the dataframe from the sparse representation to the normal dense representation (so we can treat it as regular dataframe)
    dtm_dense_named = pd.DataFrame(
        dtm_sparse.todense(), columns=vectorizer.get_feature_names_out()
    )
    print(
        "Dense matrix form:\n", dtm_dense_named.head()
    )  # take a look at dense representation
    dtm_dense_named_withid = pd.concat(
        [metadata.reset_index(), dtm_dense_named], axis=1
    )  # add back document-level covariates

    return dtm_dense_named_withid

In [10]:
## filter out na's
## for shorter runtime, random sampling of 1000
## get metadata for those
## and also renaming price col since it's likely to be corpus word
ab_small = (
    ab.loc[~ab.name.isnull(), ["id", "neighbourhood_group", "price", "name"]]
    .copy()
    .rename(columns={"price": "price_rawdata"})
    .sample(n=1000, random_state=422)
)

ab_small["name_lower"] = ab_small["name"].str.lower()
ab_small.head()

,id,neighbourhood_group,price_rawdata,name,name_lower
23821,19227560,Queens,100,Super Cozy!,super cozy!
22905,18560625,Brooklyn,30,Beautiful Private Bedroom by Prospect Park,beautiful private bedroom by prospect park
20426,16289576,Manhattan,80,Best Location on the Upper West Side! - Part II,best location on the upper west side! - part ii
2018,893413,Manhattan,2500,Architecturally Stunning Former Synagogue!,architecturally stunning former synagogue!
18790,14882137,Queens,50,"Large, beautiful room near Bushwick","large, beautiful room near bushwick"


## 2.2 Execute the dtm function to create the document-term matrix

In [11]:
## example application on raw lowercase texts;
dtm_nopre = create_dtm(
    list_of_strings=ab_small.name_lower,
    metadata=ab_small[["id", "neighbourhood_group", "price_rawdata"]],
)

Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (3, 970)>
  Coords	Values
  (0, 841)	1
  (0, 281)	1
  (1, 152)	1
  (1, 693)	1
  (1, 157)	1
  (1, 205)	1
  (1, 698)	1
  (1, 653)	1
  (2, 165)	1
  (2, 537)	1
  (2, 637)	1
  (2, 856)	1
  (2, 902)	1
  (2, 939)	1
  (2, 774)	1
  (2, 657)	1
  (2, 471)	1

Dense matrix form:
    001  10  10m  10min  10mins  1100  12mins  14  15  15min  ...  yoga  york  \
0    0   0    0      0       0     0       0   0   0      0  ...     0     0   
1    0   0    0      0       0     0       0   0   0      0  ...     0     0   
2    0   0    0      0       0     0       0   0   0      0  ...     0     0   
3    0   0    0      0       0     0       0   0   0      0  ...     0     0   
4    0   0    0      0       0     0       0   0   0      0  ...     0     0   

   you  your  yu  zen  ღღღsteps  法拉盛中心私人房間獨立衛浴  溫馨大套房  獨一無二的紐約閣樓  
0    0     0   0    0         0              0      0          0  
1    0 

In [12]:
## show first set of rows/cols
dtm_nopre.head()

## show arbitrary later cols in resulting data
dtm_nopre.shape
dtm_nopre.iloc[0:5, 480:500]

,index,id,neighbourhood_group,price_rawdata,001,10,10m,10min,10mins,1100,...,yoga,york,you,your,yu,zen,ღღღsteps,法拉盛中心私人房間獨立衛浴,溫馨大套房,獨一無二的紐約閣樓
0,23821,19227560,Queens,100,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,22905,18560625,Brooklyn,30,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20426,16289576,Manhattan,80,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2018,893413,Manhattan,2500,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,18790,14882137,Queens,50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(1000, 974)

,inclusive,incredible,incredibly,indoor,inn,inq,insane,int,interior,international,interns,invincible,inviting,inwood,island,it,italy,its,jefferson,jewel
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2.3 Use that matrix/column sums to get basic summary stats of top words

In [13]:
## summing each col
top_terms = dtm_nopre[dtm_nopre.columns[4:]].sum(axis=0)

## sorting from most frequent to least frequent
top_terms.sort_values(ascending=False)

in           367
room         244
private      163
bedroom      152
apartment    130
            ... 
gay            1
gente          1
geodesic       1
george         1
獨一無二的紐約閣樓      1
Length: 970, dtype: int64

## 2.4 Activity 2: repeat the above but using the preprocessed text data

- Stick with the same random sample of 1000 `ab_small`
- Apply the preprocessing steps from activity 1 to create a new column in `ab_small` with the preprocessed text (if you got stuck on that, try just removing stopwords)
- Use the `create_dtm` function to create a document-term matrix from the preprocessed data
- Use colsums to summarize

In [14]:
# Apply the same preprocessing to the existing 1,000-listing sample.
ab_small["name_preprocessed"] = preprocess_corpus(ab_small["name_lower"])

# Create a DTM from the preprocessed documents.
dtm_pre = create_dtm(
    list_of_strings=ab_small["name_preprocessed"],
    metadata=ab_small[["id", "neighbourhood_group", "price_rawdata"]],
)

# Rank terms by total frequency across all documents.
term_columns = dtm_pre.columns[4:]
top_terms_pre = dtm_pre[term_columns].sum().sort_values(ascending=False)
top_terms_pre.head(20)

Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 13 stored elements and shape (3, 749)>
  Coords	Values
  (0, 643)	1
  (0, 170)	1
  (1, 60)	1
  (1, 514)	1
  (1, 65)	1
  (1, 517)	1
  (1, 478)	1
  (2, 70)	1
  (2, 387)	1
  (2, 694)	1
  (2, 726)	1
  (2, 582)	1
  (2, 480)	1

Dense matrix form:
    abcd  abod  access  acidot  acogedor  across  address  ador  aesthet  \
0     0     0       0       0         0       0        0     0        0   
1     0     0       0       0         0       0        0     0        0   
2     0     0       0       0         0       0        0     0        0   
3     0     0       0       0         0       0        0     0        0   
4     0     0       0       0         0       0        0     0        0   

   afford  ...  yanke  yard  year  yellow  yoga  zen  ღღღstep  法拉盛中心私人房間獨立衛浴  \
0       0  ...      0     0     0       0     0    0        0              0   
1       0  ...      0     0     0       0     0    0        0     

room            246
privat          165
bedroom         159
cozi             97
apt              91
spaciou          88
studio           87
park             73
sunni            68
williamsburg     67
larg             64
beauti           60
near             55
east             51
heart            44
central          42
min              41
bed              40
home             37
luxuri           37
dtype: int64

# 3. Use gensim to more automatically preprocess/estimate a topic model

## 3.1 Creating the objects to feed the LDA modeling function

Different outputs described below: 
- Tokenized and preprocessed text 
- Dictionary 
- Corpus 

In [15]:
## Step 1: re-tokenize and store in list
## here, i'm doing with the raw random sample of text
## in activity, you should do with the preprocessed texts
text_raw_tokens = [wordpunct_tokenize(one_text) for one_text in ab_small.name_lower]


## Step 2: use gensim create dictionary - gets all unique words across documents
text_raw_dict = corpora.Dictionary(text_raw_tokens)
raw_len = len(text_raw_dict)  # get length for comparison below

### explore first few keys and values
### see that key is just an arbitrary counter; value is the word itself
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]}


## Step 3: filter out very rare and very common words
## here, i'm using the threshold that a word needs to appear in at least
## 5% of docs but not more than 95%
## this is an integer count of docs so i round
lower_bound = round(ab_small.shape[0] * 0.05)
upper_bound = round(ab_small.shape[0] * 0.95)

### apply filtering to dictionary
text_raw_dict.filter_extremes(no_below=lower_bound, no_above=upper_bound)
print(f"Filtering out very rare and very common words reduced the \
length of dictionary from {str(raw_len)} to {str(len(text_raw_dict))}.")
{
    k: text_raw_dict[k] for k in list(text_raw_dict)[:5]
}  # show first five entries after filtering


## Step 4: apply dictionary to TOKENIZED texts
## this creates a mapping between each word
## in a specific listing and the key in the dictionary.
## for words that remain in the filtered dictionary,
## output is a list where len(list) == n documents
## and each element in the list is a list of tuples
## containing the mappings
corpus_fromdict = [text_raw_dict.doc2bow(one_text) for one_text in text_raw_tokens]

### can apply doc2bow(one_text, return_missing = True) to print words
### eliminated from the listing bc they're not in filtered dictionary.
### but feeding that one with missing values to
### the lda function can cause errors
corpus_fromdict_showmiss = [
    text_raw_dict.doc2bow(one_text, return_missing=True) for one_text in text_raw_tokens
]
print(
    "Sample of documents represented in dictionary format (with omitted words noted):"
)
corpus_fromdict_showmiss[:10]

{0: '!', 1: 'cozy', 2: 'super', 3: 'beautiful', 4: 'bedroom'}

Filtering out very rare and very common words reduced the length of dictionary from 1047 to 31.


{0: '!', 1: 'cozy', 2: 'beautiful', 3: 'bedroom', 4: 'park'}

Sample of documents represented in dictionary format (with omitted words noted):


[([(0, 1), (1, 1)], {'super': 1}),
 ([(2, 1), (3, 1), (4, 1), (5, 1)], {'by': 1, 'prospect': 1}),
 ([(0, 1), (6, 1), (7, 1)],
  {'best': 1,
   'ii': 1,
   'location': 1,
   'on': 1,
   'part': 1,
   'side': 1,
   'upper': 1,
   'west': 1}),
 ([(0, 1)],
  {'architecturally': 1, 'former': 1, 'stunning': 1, 'synagogue': 1}),
 ([(2, 1), (8, 1), (9, 1), (10, 1), (11, 1)], {'bushwick': 1}),
 ([(4, 1), (8, 1), (9, 1), (12, 1), (13, 2)],
  {'bath': 1, 'bed': 1, 'by': 1, 'central': 1, 'college': 1, 'hunter': 1}),
 ([(9, 1), (11, 1), (14, 1), (15, 1)], {'bohemian': 1, 'brownstone': 1}),
 ([(16, 1)],
  {'fidi': 1, 'huge': 1, 'loft': 1, 'views': 1, 'w': 1, 'water': 1}),
 ([], {'hillside': 1, 'hotel': 1}),
 ([(5, 1), (9, 1), (11, 1), (14, 1), (15, 1)], {'airy': 1})]

##  3.2 Estimating the model

In [16]:
## Step 5: we're finally ready to estimate the model!
## full documentation here - https://radimrehurek.com/gensim/models/ldamodel.html
## here, we're feeding the lda function:
## (1) the corpus we created from the dictionary,
## (2) a parameter we decide on for the number of topics (k),
## (3) the dictionary itself,
## (4) parameter for number of passes through training data (more means slower), and
## (5) parameter that returns, for each word remaining in dict, the topic probabilities.
## see documentation for many other arguments you can vary
ldamod = gensim.models.ldamodel.LdaModel(
    corpus_fromdict,
    num_topics=5,
    id2word=text_raw_dict,
    passes=6,
    alpha="auto",
    per_word_topics=True,
)

print(type(ldamod))

<class 'gensim.models.ldamodel.LdaModel'>


## 3.3  Seeing what topics the estimated model discovers

In [17]:
## Post-model 1: explore corpus-wide summary of topics
### getting the topics and top words; can retrieve diff top words
topics = ldamod.print_topics(num_words=10)
for topic in topics:
    print(topic)

(0, '0.157*"in" + 0.129*"cozy" + 0.104*"the" + 0.103*"large" + 0.068*"bedroom" + 0.059*"of" + 0.054*"2" + 0.050*"room" + 0.045*"apartment" + 0.034*"apt"')
(1, '0.159*"/" + 0.120*"to" + 0.093*"in" + 0.084*"park" + 0.069*"." + 0.065*"williamsburg" + 0.065*"manhattan" + 0.057*"2" + 0.046*"apt" + 0.046*"east"')
(2, '0.144*"1" + 0.124*"!" + 0.124*"bedroom" + 0.120*"-" + 0.107*"brooklyn" + 0.068*"apt" + 0.056*"in" + 0.055*"spacious" + 0.033*"apartment" + 0.028*","')
(3, '0.194*"apartment" + 0.151*"studio" + 0.120*"beautiful" + 0.095*"and" + 0.068*"with" + 0.061*"near" + 0.040*"-" + 0.034*"in" + 0.034*"private" + 0.023*"manhattan"')
(4, '0.159*"room" + 0.148*"in" + 0.142*"," + 0.107*"private" + 0.043*"&" + 0.041*"spacious" + 0.040*"sunny" + 0.031*"bedroom" + 0.027*"with" + 0.024*"williamsburg"')


In [18]:
## Post-model 2: explore topics associated with each document
### for each item in our original dictionary, get list of topic probabilities
l = [ldamod.get_document_topics(item) for item in corpus_fromdict]
### print result
text_raw_tokens[0:5]
l[0:5]

[['super', 'cozy', '!'],
 ['beautiful', 'private', 'bedroom', 'by', 'prospect', 'park'],
 ['best',
  'location',
  'on',
  'the',
  'upper',
  'west',
  'side',
  '!',
  '-',
  'part',
  'ii'],
 ['architecturally', 'stunning', 'former', 'synagogue', '!'],
 ['large', ',', 'beautiful', 'room', 'near', 'bushwick']]

[[(0, np.float32(0.53386354)),
  (1, np.float32(0.04615056)),
  (2, np.float32(0.26808167)),
  (3, np.float32(0.04850627)),
  (4, np.float32(0.103397995))],
 [(0, np.float32(0.029008508)),
  (1, np.float32(0.027149964)),
  (2, np.float32(0.030433798)),
  (3, np.float32(0.21271767)),
  (4, np.float32(0.70069003))],
 [(0, np.float32(0.30236495)),
  (1, np.float32(0.034126073)),
  (2, np.float32(0.55224174)),
  (3, np.float32(0.03589735)),
  (4, np.float32(0.07536992))],
 [(0, np.float32(0.07623113)),
  (1, np.float32(0.07127286)),
  (2, np.float32(0.62065434)),
  (3, np.float32(0.07490866)),
  (4, np.float32(0.15693304))],
 [(0, np.float32(0.024200255)),
  (1, np.float32(0.022411928)),
  (2, np.float32(0.025050474)),
  (3, np.float32(0.5190817)),
  (4, np.float32(0.4092556))]]

### Visualizing 

In [19]:
# The raw-text visualization is intentionally omitted.
# Activity 3 below visualizes the preprocessed model instead.

## 3.4 Activity 3

- Preprocess the texts if you haven't already
- Run the topic model with preprocessed texts
- Play around with other parameters like `n_topics` to find a configuration that produces useful topics

If you get stuck on the preprocessing part, you can use below function and example code for applying it. Then continue as above (start with tokenizing).

In [20]:
# Tokenize the already-preprocessed listing names.
text_preprocessed_tokens = [text.split() for text in ab_small["name_preprocessed"]]

# Build and filter a document dictionary.
text_preprocessed_dict = corpora.Dictionary(text_preprocessed_tokens)
preprocessed_len = len(text_preprocessed_dict)

# Remove words that are too rare or occur in almost every document.
text_preprocessed_dict.filter_extremes(
    no_below=max(2, round(len(text_preprocessed_tokens) * 0.05)),
    no_above=0.95,
)
print(
    f"Dictionary terms: {preprocessed_len} -> "
    f"{len(text_preprocessed_dict)} after filtering."
)

# Represent each document as (term_id, count) pairs.
corpus_preprocessed_fromdict = [
    text_preprocessed_dict.doc2bow(tokens) for tokens in text_preprocessed_tokens
]

# Estimate a reproducible five-topic LDA model.
ldamod_preprocessed = gensim.models.ldamodel.LdaModel(
    corpus=corpus_preprocessed_fromdict,
    num_topics=5,
    id2word=text_preprocessed_dict,
    passes=10,
    alpha="symmetric",
    random_state=422,
)

# Inspect the top words associated with each topic.
preprocessed_topics = ldamod_preprocessed.print_topics(num_words=10)
for topic_number, topic_words in preprocessed_topics:
    print(f"Topic {topic_number}: {topic_words}")

# Visualize the preprocessed model (so filtered stopwords do not appear).
lda_display_preprocessed = gensimvis.prepare(
    ldamod_preprocessed,
    corpus_preprocessed_fromdict,
    text_preprocessed_dict,
)


# pyLDAvis may produce zero-imaginary-component complex values.
def _real_if_complex(value):
    if np.iscomplexobj(value) and np.allclose(np.imag(value), 0):
        return np.real(value)
    return value


_sanitized_frames = {
    attribute: getattr(lda_display_preprocessed, attribute).map(_real_if_complex)
    for attribute in ["topic_coordinates", "topic_info", "token_table"]
}
lda_display_preprocessed = lda_display_preprocessed._replace(**_sanitized_frames)
pyLDAvis.display(lda_display_preprocessed)

Dictionary terms: 749 -> 14 after filtering.
Topic 0: 0.346*"room" + 0.266*"privat" + 0.200*"bedroom" + 0.134*"spaciou" + 0.032*"near" + 0.017*"larg" + 0.001*"park" + 0.001*"cozi" + 0.001*"east" + 0.001*"sunni"
Topic 1: 0.320*"apt" + 0.280*"park" + 0.162*"cozi" + 0.104*"near" + 0.100*"bedroom" + 0.017*"sunni" + 0.005*"privat" + 0.005*"east" + 0.003*"studio" + 0.001*"room"
Topic 2: 0.439*"studio" + 0.301*"cozi" + 0.121*"room" + 0.080*"privat" + 0.037*"near" + 0.010*"williamsburg" + 0.002*"apt" + 0.002*"spaciou" + 0.002*"east" + 0.001*"bedroom"
Topic 3: 0.288*"beauti" + 0.265*"larg" + 0.246*"east" + 0.120*"bedroom" + 0.027*"near" + 0.025*"park" + 0.014*"studio" + 0.009*"spaciou" + 0.001*"apt" + 0.001*"privat"
Topic 4: 0.337*"williamsburg" + 0.330*"sunni" + 0.161*"room" + 0.070*"apt" + 0.064*"spaciou" + 0.011*"beauti" + 0.010*"privat" + 0.009*"larg" + 0.003*"bedroom" + 0.001*"near"
